# BRONZE TABLES

In [0]:
from pyspark.errors import PySparkException

# Create a maven_toys catalog or use the default workspace catalog.
# Upload the dataset to a catalog to be called in the load function.
spark.sql(f"USE CATALOG maven_toys")

def load_bronze():
    """This Python function loads the raw data into the bronze schema. It will use the currently selected catalog, drop the bronze schema and the tables using the cascade keyword. Then recreate the bronze schema and load the schema with tables of the source data. The table loading is done using CTAs in SQL.

    Parameters: 
        None

    Usage: 
        load_bronze()
    """
    try:
        #  Drop the schema along with the tables if it exists and create a new schema
        spark.sql(f"DROP SCHEMA IF EXISTS bronze CASCADE;")
        spark.sql(f"CREATE SCHEMA bronze;")

        #  Create the table bronze.raw_orders using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_orders
        AS
        SELECT order_id, created_at, website_session_id, user_id, primary_product_id, items_purchased, price_usd, cogs_usd
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/orders.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")

        #  Create the table bronze.raw_order_items using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_order_items
        AS
        SELECT order_item_id, created_at, order_id, product_id, is_primary_item, price_usd, cogs_usd
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/order_items.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")

        #  Create the table bronze.raw_order_item_refunds using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_order_item_refunds
        AS
        SELECT order_item_refund_id, created_at, order_item_id, order_id, refund_amount_usd
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/order_item_refunds.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")

        #  Create the table bronze.raw_products using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_products
        AS
        SELECT product_id, created_at, product_name
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/products.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")

        #  Create the table bronze.raw_website_sessions using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_website_sessions
        AS
        SELECT website_session_id,created_at,user_id,is_repeat_session,utm_source,utm_campaign,utm_content,device_type,http_referer
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/website_sessions.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")

        #  Create the table bronze.raw_website_pageviews using CTAS
        spark.sql(f"""
        CREATE TABLE bronze.raw_website_pageviews
        AS
        SELECT website_pageview_id,created_at,website_session_id,pageview_url
        FROM read_files(
        '/Volumes/maven_toys/default/maven_data/website_pageviews.csv',
        format => 'csv',
        header => true,
        inferSchema => true
        );""")
    except PySparkException as ex:
        print("Error Condition   : " + ex.getErrorClass())
        print("Message arguments : " + str(ex.getMessageParameters()))
        print("SQLSTATE          : " + ex.getSqlState())
        print(ex)

load_bronze()
spark.sql(f"SHOW TABLES IN bronze").display()

# SILVER TABLES

In [0]:
# Use your created Catalog
spark.sql(f"USE CATALOG maven_toys")

def load_silver():
    """This Python function loads the cleaned data into the silver schema. It will use the currently selected catalog, drop the silver schema and the tables using the cascade keyword. Then recreate the silver schema and load the schema with tables of the cleaned data. The table loading is done using CTAs in SQL.

    Parameters: 
        None

    Usage: 
        load_silver()
    """
    try:
        #  Drop the schema along with the tables if it exists and create a new schema
        spark.sql(f"DROP SCHEMA IF EXISTS silver CASCADE;")
        spark.sql(f"CREATE SCHEMA silver;")

        #  Create the table silver.orders using CTAS
        spark.sql(f"""
        CREATE TABLE silver.orders
        AS
        SELECT 
            order_id, 
            created_at, 
            website_session_id, 
            user_id, 
            primary_product_id, 
            items_purchased, 
            price_usd, 
            cogs_usd
        FROM bronze.raw_orders;""")

        #  Create the table silver.order_items using CTAS
        spark.sql(f"""
        CREATE TABLE silver.order_items
        AS
        SELECT 
            order_item_id, 
            created_at, 
            order_id, 
            product_id, 
            is_primary_item, 
            price_usd, 
            cogs_usd
        FROM bronze.raw_order_items;""")

        #  Create the table silver.order_item_refunds using CTAS
        spark.sql(f"""
        CREATE TABLE silver.order_item_refunds
        AS
        SELECT 
            order_item_refund_id, 
            created_at, 
            order_item_id,
            order_id,
            refund_amount_usd
        FROM bronze.raw_order_item_refunds;""")

        #  Create the table silver.products using CTAS
        spark.sql(f"""
        CREATE TABLE silver.products
        AS
        SELECT 
            product_id,
            created_at,
            product_name
        FROM bronze.raw_products;""")

        #  Create the table silver.website_sessions using CTAS
        spark.sql(f"""
        CREATE TABLE silver.website_sessions
        AS
        SELECT 
            website_session_id,
            created_at,
            user_id,
            is_repeat_session,
            utm_source,
            utm_campaign,
            utm_content,
            device_type,
            http_referer
        FROM bronze.raw_website_sessions;""")

        #  Create the table silver.website_pageviews using CTAS
        spark.sql(f"""
        CREATE TABLE silver.website_pageviews
        AS
        SELECT 
            website_pageview_id,
            created_at,
            website_session_id,
            pageview_url
        FROM bronze.raw_website_pageviews;""")
    except PySparkException as ex:
        print("Error Condition   : " + ex.getErrorClass())
        print("Message arguments : " + str(ex.getMessageParameters()))
        print("SQLSTATE          : " + ex.getSqlState())
        print(ex)

# Call the function and display current schema tables
load_silver()
spark.sql(f"SHOW TABLES IN silver").display()

## QUALITY CHECKS

In [0]:
"""These are quality test checks to ensure the data is clean and ready for analysis. 
These tests include checking for:
    - Null or duplicate primary keys.
    - Unwanted spaces in string fields.
    - Data standardization and consistency.
    - Invalid date ranges
    
Each of these tests is handled by a Python function written in SQL. Using flexible variables, the function takes the column name and the table name as inputs to run the test on specific tables and columns. This makes for simple yet efficient and scalable testing.

Parameters:
    value/key/date: The column name as a str data type
    table: The table name as a str data type

Usage:
    no_duplicates('order_id', 'orders')
    unwanted_spaces('product_name', 'products')
    stan_check('product_name', 'products')
    negative_cost('price_usd', 'order_items')
    invalid_date('created_at', 'website_pageviews')"""

# Check for NULLs or Duplicates in Primary Key
# Expectation: No Results
def no_duplicates(key: str, table: str):
    return spark.sql(f"""
    SELECT 
        {key},
        COUNT(*) 
    FROM silver.{table}
    GROUP BY {key}
    HAVING COUNT(*) > 1 OR {key} IS NULL;""")

# Check for Unwanted Spaces
# Expectation: No Results
def unwanted_spaces(value: str, table: str):
    return spark.sql(f"""
    SELECT 
        {value}
    FROM silver.{table}
    WHERE {value} != TRIM({value});""")

# Data Standardization & Consistency
def stan_check(value: str, table: str):
    return spark.sql(f"""
    SELECT DISTINCT
        {value}
    FROM silver.{table};""")

# Check for NULLs or Negative Values in Cost
# Expectation: No Results
def negative_cost(value: str, table: str):
    return spark.sql(f"""
    SELECT 
        {value}
    FROM silver.{table}
    WHERE {value} < 0 OR {value} IS NULL;""")

# Check for Invalid Dates (for timestamp columns)
# Expectation: No Invalid Dates
def invalid_date(date: str, table: str):
    return spark.sql(f"""
    SELECT
        {date}
    FROM silver.{table}
    WHERE {date} IS NULL
    OR {date} < '1900-01-01'
    OR {date} > '2050-01-01';""")

# GOLD MODELS

In [0]:
# Use your created Catalog
spark.sql(f"USE CATALOG maven_toys")

def load_gold():
    """This Python function transforms and combines data from the Silver layer, producing SQL views that are aggregation ready and offer insight into valuable business metrics. It will use the currently selected catalog, drop the gold schema and the tables using the cascade keyword. Then recreate the gold schema and load the schema with tables using CTAs of specifically catered and transformed SQL queries.

    Parameters: 
        None

    Usage: 
        load_gold()
    """
    try:
        #  Drop the schema along with the views if it exists and create a new schema
        spark.sql(f"DROP SCHEMA IF EXISTS gold CASCADE;")
        spark.sql(f"CREATE SCHEMA gold;")

        # Create the view gold.fact_orders using CVAs
        spark.sql(f"""
        CREATE VIEW gold.fact_orders 
        AS
        SELECT 
            order_id,
            created_at,
            user_id,
            primary_product_id,
            items_purchased,
            price_usd,
            cogs_usd
        FROM silver.orders;""")

        # Create the view gold.dim_order_items using CVAs
        spark.sql(f"""
        CREATE VIEW gold.dim_order_items 
        AS
        SELECT 
            oi.order_item_id AS item_id,
            DATE_FORMAT(oi.created_at, 'yyyy-MM-dd')  AS created_on,
            oi.order_id,
            p.product_name,
            oi.is_primary_item,
            oi.price_usd,
            oi.cogs_usd,
            CASE
                WHEN oi.order_item_id = r.order_item_id THEN 1
                ELSE 0
            END as is_refunded,
            DATE_FORMAT(r.created_at, 'yyyy-MM-dd') AS refund_created_on
        FROM silver.order_items oi
        LEFT JOIN silver.products p
        ON oi.product_id = p.product_id
        LEFT JOIN silver.order_item_refunds r
        ON oi.order_item_id = r.order_item_id;""")

        # Create the view gold.dim_sessions using CVAs
        spark.sql(f"""
        CREATE VIEW gold.dim_sessions 
        AS
        WITH CTE_MostRecentPageview AS
        (
            SELECT
            website_session_id,
            LAST_VALUE(pageview_url) AS last_url,
            LAST_VALUE(created_at) AS last_visit_at
            FROM silver.website_pageviews
            GROUP BY website_session_id
        )
        SELECT DISTINCT
            ws.website_session_id AS session_id,
            ws.created_at,
            ws.user_id,
            ws.is_repeat_session,
            ws.utm_source AS traffic_origin,
            ws.utm_campaign AS marketing_campaign,
            ws.utm_content AS ad_variant,
            ws.device_type,
            mrp.last_url,
            mrp.last_visit_at
        FROM silver.website_sessions ws
        LEFT JOIN silver.website_pageviews wp
        ON ws.website_session_id = wp.website_session_id
        LEFT JOIN CTE_MostRecentPageview mrp
        ON ws.website_session_id = mrp.website_session_id
        ORDER BY session_id;""")
    except PySparkException as ex:
        print("Error Condition   : " + ex.getErrorClass())
        print("Message arguments : " + str(ex.getMessageParameters()))
        print("SQLSTATE          : " + ex.getSqlState())
        print(ex)

# Call the function and display current schema tables
load_gold()
spark.sql(f"SHOW VIEWS IN gold").display()

## REPORTS

In [0]:
%sql
--Monthly Revenue
SELECT
    DATE_PART('month', created_at) AS month,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price_usd), 2) AS revenue
FROM gold.fact_orders
GROUP BY DATE_PART('month', created_at)
ORDER BY DATE_PART('month', created_at)

In [0]:
%sql
--Product Performance
SELECT
    product_name,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(price_usd), 2) AS revenue,
    ROUND(SUM(price_usd) - SUM(cogs_usd), 2) AS profit,
    -- Total refunds per product
    SUM(is_refunded) AS total_refunds
FROM gold.dim_order_items
GROUP BY product_name
ORDER BY total_orders DESC

In [0]:
%sql
--UTM Source Performance (Which source of traffic was the most effective?)
SELECT
    traffic_origin,
    COUNT(DISTINCT session_id) AS total_sessions
FROM gold.dim_sessions
GROUP BY traffic_origin
ORDER BY total_sessions DESC